In [1]:
# ============================================================
# RENABAP · Análisis 1 — Universo y evolución (AMT)
# BLOQUE 1: Carga + armonización + geometría
# Entorno: Google Colab | Fuente de datos: repo GitHub Camilamop/RENABAP
# ============================================================
import os
REPO_URL = "https://github.com/Camilamop/RENABAP.git"

if os.path.exists('/content/RENABAP'):
    !cd /content/RENABAP && git fetch origin && git reset --hard origin/master
else:
    !git clone {REPO_URL} /content/RENABAP

!pip install geopandas --quiet

# chequeo rápido: ¿el 2023 ya trae WKT?
import pandas as pd
cols = pd.read_csv('/content/RENABAP/data/raw/RENABAP_2023.csv', nrows=0).columns.tolist()
print("¿WKT en 2023?", "WKT" in cols, "| columnas:", len(cols))

Cloning into '/content/RENABAP'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 100 (delta 24), reused 83 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 7.64 MiB | 13.04 MiB/s, done.
Resolving deltas: 100% (24/24), done.
¿WKT en 2023? True | columnas: 19


In [2]:
!pip install geopandas --quiet

In [3]:
import pandas as pd
import geopandas as gpd
from shapely import wkt as shapely_wkt
from pathlib import Path # Add this import

RAIZ = Path("/content/RENABAP") # Convert RAIZ to a Path object
DIR_DATOS  = RAIZ / "data" / "raw"
DIR_SALIDA = RAIZ / "data" / "processed"
DIR_SALIDA.mkdir(parents=True, exist_ok=True)

CRS_ORIGEN = "EPSG:4326"

In [4]:
# ============================================================
# Diccionario de armonización + esquema núcleo + ARCHIVOS
# nombre_original -> nombre_núcleo
# ============================================================
MAPEO = {
    2018: {
        "id_renabap": "id_renabap",
        "Nombre del barrio": "nombre_barrio",
        "Nombre de provincia": "provincia",
        "Nombre de departamentos/comuna": "departamento",
        "Localidad": "localidad",
        "Cantidad de familias": "cant_familias",
        "Tamaño (km2)": "superficie_km2",
        "Año de creación": "anio_creacion",
        "WKT": "wkt",
    },
    2022: {
        "renabap_id": "id_renabap",
        "nombre_barrio": "nombre_barrio",
        "provincia": "provincia",
        "departamento": "departamento",
        "localidad": "localidad",
        "cantidad_familias_aproximada": "cant_familias",
        "superficie_m2": "superficie_m2",
        "decada_de_creacion": "decada_creacion",
        "WKT": "wkt",
    },
    2023: {
        "id_renabap": "id_renabap",
        "nombre_barrio": "nombre_barrio",
        "provincia": "provincia",
        "departamento": "departamento",
        "localidad": "localidad",
        "cantidad_familias_aproximada": "cant_familias",
        "superficie_m2": "superficie_m2",
        "decada_de_creacion": "decada_creacion",
        "WKT": "wkt",
    },
}

NUCLEO = [
    "id_renabap", "registro", "nombre_barrio", "provincia", "departamento",
    "localidad", "cant_familias", "superficie_km2",
    "anio_creacion", "decada_creacion", "wkt",
]

ARCHIVOS = {
    2018: DIR_DATOS / "RENABAP_2018.csv",
    2022: DIR_DATOS / "RENABAP_2022.csv",
    2023: DIR_DATOS / "RENABAP_2023.csv",
}

# chequeo temprano: avisar si falta algún archivo
for anio, ruta in ARCHIVOS.items():
    print(f"  {anio}: {'OK' if ruta.exists() else 'FALTA -> ' + str(ruta)}")



  2018: OK
  2022: OK
  2023: OK


In [5]:
# ============================================================
# Funciones de carga y armonización
# ============================================================
def cargar_crudo(anio: int, ruta: Path) -> pd.DataFrame:
    df = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8")
    df.columns = [c.strip() for c in df.columns]   # limpia espacios en headers
    return df


def normalizar_decada(serie_cruda: pd.Series) -> pd.Series:
    """Extrae el año de textos tipo 'Década 1990' y lo deja como entero (1990)."""
    s = serie_cruda.astype(str).str.extract(r"(\d{4})")[0]
    return pd.to_numeric(s, errors="coerce").astype("Int64")


def armonizar(anio: int, df: pd.DataFrame) -> pd.DataFrame:
    mapa = MAPEO[anio]

    cols_presentes = {orig: nuevo for orig, nuevo in mapa.items() if orig in df.columns}
    out = df[list(cols_presentes.keys())].rename(columns=cols_presentes).copy()

    # registro (edición del registro)
    out["registro"] = anio

    # id como texto limpio
    out["id_renabap"] = out["id_renabap"].str.strip()

    # superficie a km2 (2022 y 2023 vienen en m2)
    if "superficie_m2" in out.columns:
        out["superficie_km2"] = pd.to_numeric(out["superficie_m2"], errors="coerce") / 1_000_000
        out = out.drop(columns=["superficie_m2"])
    elif "superficie_km2" in out.columns:
        out["superficie_km2"] = pd.to_numeric(out["superficie_km2"], errors="coerce")

    # cantidad de familias -> numérico
    if "cant_familias" in out.columns:
        out["cant_familias"] = pd.to_numeric(out["cant_familias"], errors="coerce").astype("Int64")

    # año de creación -> numérico (solo 2018 lo trae confiable)
    if "anio_creacion" in out.columns:
        anio_num = pd.to_numeric(out["anio_creacion"], errors="coerce")
        anio_num = anio_num.where((anio_num >= 1800) & (anio_num <= 2025))  # descarta 0 y basura
        out["anio_creacion"] = anio_num.astype("Int64")
    else:
        out["anio_creacion"] = pd.array([pd.NA] * len(out), dtype="Int64")


    if "decada_creacion" in out.columns:
        out["decada_creacion"] = normalizar_decada(out["decada_creacion"])
    else:
        out["decada_creacion"] = (out["anio_creacion"] // 10 * 10).astype("Int64")

    # localidad: garantizar la columna aunque alguna edición no la trajera
    if "localidad" not in out.columns:
        out["localidad"] = pd.NA

    # reordenar a esquema núcleo (agrega faltantes como NA, p.ej. 'wkt' en 2023)
    for c in NUCLEO:
        if c not in out.columns:
            out[c] = pd.NA
    return out[NUCLEO]


def a_geodataframe(df: pd.DataFrame) -> gpd.GeoDataFrame:
    """Parsea el WKT a geometría y arma un GeoDataFrame en EPSG:4326."""
    geom = df["wkt"].apply(lambda w: shapely_wkt.loads(w) if isinstance(w, str) and w.strip() else None)
    gdf = gpd.GeoDataFrame(df.drop(columns=["wkt"]), geometry=geom, crs=CRS_ORIGEN)
    return gdf


In [6]:
# ============================================================
# Ejecución
# ============================================================
def main():
    piezas = []
    print("Cargando y armonizando los tres registros...\n")
    for anio, ruta in ARCHIVOS.items():
        crudo = cargar_crudo(anio, ruta)
        arm = armonizar(anio, crudo)
        piezas.append(arm)
        n_geom = arm["wkt"].apply(lambda w: isinstance(w, str) and w.strip() != "").sum()
        print(f"  {anio}: {len(arm):>5} barrios | geometrías no vacías: {n_geom}")

    maestra = pd.concat(piezas, ignore_index=True)
    gdf = a_geodataframe(maestra)
    invalidas = gdf.geometry.isna().sum()

    print(f"\nTabla maestra (largo): {len(maestra)} filas "
          f"{maestra['registro'].value_counts().sort_index().to_dict()}")
    print(f"Geometrías que no parsearon o faltan: {invalidas}")

    # CSV de atributos (sin geometría)
    maestra.drop(columns=["wkt"]).to_csv(DIR_SALIDA / "renabap_maestra_largo.csv", index=False)

    # FIX: escritura robusta del GPKG por capas (sin pisar capas ni romper con capa vacía)
    gpkg_path = DIR_SALIDA / "renabap_nacional.gpkg"
    if gpkg_path.exists():
        gpkg_path.unlink()
    primera = True
    for anio in ARCHIVOS:
        sub = gdf[(gdf["registro"] == anio) & gdf.geometry.notna()]
        if len(sub) == 0:
            print(f"  [AVISO] {anio}: sin geometrías -> no se escribe capa en el GPKG.")
            continue
        sub.to_file(gpkg_path, layer=f"renabap_{anio}", driver="GPKG",
                    mode="w" if primera else "a")
        primera = False

    print(f"\nExportado en: {DIR_SALIDA.resolve()}")
    print("  - renabap_maestra_largo.csv")
    print("  - renabap_nacional.gpkg (una capa por edición con geometría disponible)")

    print("\n--- Familias totales por registro (nacional) ---")
    print(maestra.groupby("registro")["cant_familias"].agg(["count", "sum"]))
    print("\n--- Cobertura de 'década de creación' por registro ---")
    print(maestra.groupby("registro")["decada_creacion"].apply(lambda s: round(s.notna().mean(), 3)))
    print("\n--- Cobertura de 'localidad' por registro ---")
    print(maestra.groupby("registro")["localidad"].apply(
        lambda s: round((s.notna() & (s.astype(str).str.strip() != "")).mean(), 3)))

    # aviso explícito si algún año quedó sin geometría (hoy: 2023)
    sin_geo = [a for a in ARCHIVOS if gdf[(gdf["registro"] == a) & gdf.geometry.notna()].empty]
    if sin_geo:
        print("\n" + "="*60)
        print(f"⚠️  SIN GEOMETRÍA: {sin_geo}. Para el recorte AMT y los mapas")
        print("    necesitás la versión con polígonos de esa(s) edición(es)")
        print("    (GeoJSON/SHP/GPKG oficial o CSV con columna WKT) en data/raw/.")
        print("="*60)

    return maestra, gdf


maestra, gdf = main()


Cargando y armonizando los tres registros...

  2018:  4416 barrios | geometrías no vacías: 4416
  2022:  5687 barrios | geometrías no vacías: 5687
  2023:  6467 barrios | geometrías no vacías: 6467

Tabla maestra (largo): 16570 filas {2018: 4416, 2022: 5687, 2023: 6467}
Geometrías que no parsearon o faltan: 0

Exportado en: /content/RENABAP/data/processed
  - renabap_maestra_largo.csv
  - renabap_nacional.gpkg (una capa por edición con geometría disponible)

--- Familias totales por registro (nacional) ---
          count      sum
registro                
2018       4416   925609
2022       5687  1165275
2023       6467  1237795

--- Cobertura de 'década de creación' por registro ---
registro
2018    0.679
2022    1.000
2023    1.000
Name: decada_creacion, dtype: float64

--- Cobertura de 'localidad' por registro ---
registro
2018    1.0
2022    1.0
2023    1.0
Name: localidad, dtype: float64


In [7]:
# ============================================================
# Imports, rutas y parámetros
# ============================================================
import re
import pandas as pd
import geopandas as gpd
from pathlib import Path

RAIZ       = Path("/content/RENABAP")
DIR_PROC   = RAIZ / "data" / "processed"
DIR_QGIS   = DIR_PROC / "qgis"                       # tus recortes manuales (corroboración)
GPKG_NAC   = DIR_PROC / "renabap_nacional.gpkg"      # <- salida del Bloque 1

# --- PARÁMETRO A EDITAR: dónde está tu polígono límite del AMT ---
LIMITE_PATH = RAIZ / "data" / "raw" / "limite_amt.gpkg"

ANIOS      = [2018, 2022, 2023]
CRS_DATOS  = "EPSG:4326"   # almacenamiento / recorte
CRS_METRICO = "EPSG:5345"  # POSGAR 2007 Faja 3 (Tucumán) -> solo para áreas/distancias

# Conteos de tus recortes de QGIS (referencia de corroboración)
QGIS_REF = {2018: 144, 2022: 170, 2023: 213}

# --- Chequeos previos, con mensajes claros ---
assert GPKG_NAC.exists(), (
    f"No encuentro {GPKG_NAC}. Corré primero el Bloque 1 en esta sesión."
)
if not LIMITE_PATH.exists():
    raise FileNotFoundError(
        f"Falta el polígono límite en {LIMITE_PATH}.\n"
        "  -> Subilo al repo como data/raw/limite_amt.gpkg y re-corré la celda de refresco,\n"
        "     o cambiá LIMITE_PATH a la ruta donde lo tengas (por ej. /content/limite_amt.gpkg)."
    )

print("Capas en renabap_nacional.gpkg:", gpd.list_layers(GPKG_NAC)["name"].tolist())
print("Límite AMT:", LIMITE_PATH.name, "-> OK")


Capas en renabap_nacional.gpkg: ['renabap_2018', 'renabap_2022', 'renabap_2023']
Límite AMT: limite_amt.gpkg -> OK


In [8]:
# ============================================================
# Cargar el límite, disolver a una sola geometría y alinear CRS
# ============================================================
limite = gpd.read_file(LIMITE_PATH)
if limite.crs is None:
    raise ValueError("El límite no tiene CRS definido. Asignalo en QGIS antes de exportar (idealmente EPSG:4326).")

limite = limite.to_crs(CRS_DATOS)

# geometría única (marco fijo). union_all() en versiones nuevas; unary_union en las previas.
try:
    LIMITE_GEOM = limite.geometry.union_all()
except AttributeError:
    LIMITE_GEOM = limite.geometry.unary_union

area_km2 = gpd.GeoSeries([LIMITE_GEOM], crs=CRS_DATOS).to_crs(CRS_METRICO).area.iloc[0] / 1e6
print(f"Límite disuelto: {LIMITE_GEOM.geom_type} | superficie ≈ {area_km2:,.1f} km² (medida en {CRS_METRICO})")


Límite disuelto: Polygon | superficie ≈ 532.5 km² (medida en EPSG:5345)


In [9]:
# ============================================================
# Recorte por 'intersecta' (barrio entero) sobre las tres fotos
# ============================================================
GPKG_AMT = DIR_PROC / "renabap_amt.gpkg"
if GPKG_AMT.exists():
    GPKG_AMT.unlink()

piezas = []
primera = True
print("Recortando al AMT (regla: intersecta, barrio entero)\n")
for anio in ANIOS:
    capa = gpd.read_file(GPKG_NAC, layer=f"renabap_{anio}").to_crs(CRS_DATOS)
    sub = capa[capa.intersects(LIMITE_GEOM)].copy()

    ref = QGIS_REF.get(anio)
    marca = "✓" if ref is not None and len(sub) == ref else ("~" if ref else " ")
    print(f"  {anio}: {len(sub):>4} barrios en el AMT   (QGIS={ref})  {marca}")

    piezas.append(sub.drop(columns="geometry"))
    if len(sub):
        sub.to_file(GPKG_AMT, layer=f"renabap_amt_{anio}", driver="GPKG",
                    mode="w" if primera else "a")
        primera = False

maestra_amt = pd.concat(piezas, ignore_index=True)
maestra_amt.to_csv(DIR_PROC / "renabap_amt_maestra_largo.csv", index=False)

print(f"\nExportado en {DIR_PROC.resolve()}:")
print("  - renabap_amt.gpkg (capas renabap_amt_2018/2022/2023)")
print("  - renabap_amt_maestra_largo.csv")


Recortando al AMT (regla: intersecta, barrio entero)

  2018:  144 barrios en el AMT   (QGIS=144)  ✓
  2022:  170 barrios en el AMT   (QGIS=170)  ✓
  2023:  211 barrios en el AMT   (QGIS=213)  ~

Exportado en /content/RENABAP/data/processed:
  - renabap_amt.gpkg (capas renabap_amt_2018/2022/2023)
  - renabap_amt_maestra_largo.csv


In [10]:
# ============================================================
# Corroboración automática contra tus recortes de QGIS
#   - compara conteo
#   - compara el CONJUNTO de ids (qué barrios sobran/faltan)
# ============================================================
def columna_id(df: pd.DataFrame):
    """Encuentra la columna de id_renabap sin importar mayúsculas/espacios/guiones."""
    for c in df.columns:
        if re.sub(r'[^a-z]', '', c.lower()) == "idrenabap":
            return c
    return None

print("Corroboración notebook vs QGIS (regla intersecta)\n")
for anio in ANIOS:
    ruta_q = DIR_QGIS / f"RENABAP_AMT_{anio}.csv"
    ids_nb = set(maestra_amt.loc[maestra_amt["registro"] == anio, "id_renabap"].astype(str).str.strip())

    if not ruta_q.exists():
        print(f"  {anio}: notebook={len(ids_nb)} | (no encuentro el recorte QGIS)")
        continue

    q = pd.read_csv(ruta_q, dtype=str, keep_default_na=False)
    q.columns = [c.strip() for c in q.columns]
    ic = columna_id(q)
    if ic is None:
        print(f"  {anio}: notebook={len(ids_nb)} | QGIS={len(q)} | (QGIS sin columna id -> comparo solo conteo)")
        continue

    ids_q = set(q[ic].astype(str).str.strip())
    solo_nb = ids_nb - ids_q      # barrios que el notebook incluye y QGIS no
    solo_q  = ids_q - ids_nb      # barrios que QGIS incluye y el notebook no
    print(f"  {anio}: notebook={len(ids_nb)} | QGIS={len(ids_q)} | "
          f"solo_notebook={len(solo_nb)} | solo_QGIS={len(solo_q)}")
    if solo_nb:
        print(f"        ej. solo_notebook: {sorted(solo_nb)[:5]}")
    if solo_q:
        print(f"        ej. solo_QGIS:     {sorted(solo_q)[:5]}")

print("\nLectura: solo_QGIS>0 = barrios que tu polígono capturó y el notebook no (revisar borde/CRS).")
print("         solo_notebook>0 = barrios de borde que 'intersecta' suma (esperable si en QGIS usaste 'clip' o recortaste a mano).")


Corroboración notebook vs QGIS (regla intersecta)

  2018: notebook=144 | QGIS=144 | solo_notebook=0 | solo_QGIS=0
  2022: notebook=170 | QGIS=170 | solo_notebook=0 | solo_QGIS=0
  2023: notebook=211 | QGIS=213 | solo_notebook=0 | solo_QGIS=2
        ej. solo_QGIS:     ['6859', '6861']

Lectura: solo_QGIS>0 = barrios que tu polígono capturó y el notebook no (revisar borde/CRS).
         solo_notebook>0 = barrios de borde que 'intersecta' suma (esperable si en QGIS usaste 'clip' o recortaste a mano).


In [11]:
# ============================================================
# Comparabilidad temporal + resumen del AMT
# ============================================================
print("--- Localidades presentes por foto (dentro del AMT) ---")
sets = {}
for anio in ANIOS:
    locs = set(
        maestra_amt.loc[maestra_amt["registro"] == anio, "localidad"]
        .dropna().astype(str).str.strip()
    )
    locs.discard("")
    sets[anio] = locs
    print(f"  {anio}: {len(locs)} localidades")

# ¿el conjunto de localidades cambia entre fotos? (clave para la comparación 2016/18/21)
base = sets[ANIOS[0]]
for anio in ANIOS[1:]:
    nuevas = sets[anio] - base
    perdidas = base - sets[anio]
    print(f"  {ANIOS[0]}→{anio}: +{len(nuevas)} nuevas / -{len(perdidas)} ausentes",
          f"| nuevas: {sorted(nuevas)}" if nuevas else "")

print("\n--- Resumen por registro (AMT) ---")
resumen = maestra_amt.groupby("registro").agg(
    barrios=("id_renabap", "count"),
    familias=("cant_familias", "sum"),
    superficie_km2=("superficie_km2", "sum"),
)
print(resumen)
print("\nRecordá: parte del crecimiento entre fotos es ampliación de cobertura del registro, no fenómeno real (declararlo).")


--- Localidades presentes por foto (dentro del AMT) ---
  2018: 15 localidades
  2022: 18 localidades
  2023: 20 localidades
  2018→2022: +4 nuevas / -1 ausentes | nuevas: ['Colombres', 'Delfín Gallo', 'Ingenio La Florida', 'Los Nogales']
  2018→2023: +6 nuevas / -1 ausentes | nuevas: ['Colombres', 'Delfín Gallo', 'Ingenio La Florida', 'Lastenia', 'Los Nogales', 'Ranchillos']

--- Resumen por registro (AMT) ---
          barrios  familias  superficie_km2
registro                                   
2018          144     30681       10.930252
2022          170     36457       12.559957
2023          211     38009       13.532202

Recordá: parte del crecimiento entre fotos es ampliación de cobertura del registro, no fenómeno real (declararlo).


In [ ]:
import pandas as pd
import unicodedata


def normalizar(x):
    """minúscula, sin acentos y sin espacios extra — para poder unir por nombre."""
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    x = "".join(c for c in unicodedata.normalize("NFKD", x) if not unicodedata.combining(c))
    return x



In [ ]:
# CELDA 1 — Localidades nuevas (salida de tu diferencia de conjuntos)
nuevas = pd.DataFrame({
    "localidad": ["Colombres", "Delfín Gallo", "Ingenio La Florida",
                  "Los Nogales", "Lastenia", "Ranchillos"],
    "primer_registro": [2022, 2022, 2022, 2022, 2023, 2023],
})

indec_nombre = {
    "Colombres": "Colombres",
    "Delfín Gallo": "Delfín Gallo",
    "Ingenio La Florida": "La Florida",          # INDEC la registra como "La Florida" (incluye el Ingenio)
    "Los Nogales": "Los Nogales",
    "Lastenia": "Lastenia",                      # verificar: puede figurar dentro de otro gobierno local
    "Ranchillos": "Ranchillos y San Miguel",     # nombre completo del gobierno local en INDEC
}
poblacion_2022 = {
    "Colombres": 9781,
    "Delfín Gallo": 11189,
    "Ingenio La Florida": None,   # "La Florida": 5.959 en 2010 -> completar con 2022
    "Los Nogales": 7987,
    "Lastenia": None,             # completar (ver nota de matching)
    "Ranchillos": 14242,
}



In [ ]:
# CELDA 4 — Armar la tabla, clasificar por rango y anotar caveats
def rango(p):
    if pd.isna(p):
        return "sin dato"
    if p < 2000:
        return "< 2.000"
    if p <= 10000:
        return "2.000–10.000"
    return "> 10.000"

nuevas["gobierno_local_INDEC"] = nuevas["localidad"].map(indec_nombre)
nuevas["poblacion_2022"] = nuevas["localidad"].map(poblacion_2022)
nuevas["rango_tamano"] = nuevas["poblacion_2022"].map(rango)

# Nota de interpretación (unidad territorial / aglomerado):
notas = {
    "Delfín Gallo": "Integra el aglomerado Delfín Gallo–Colombres–La Florida; pob. de gobierno local ≠ pob. de aglomerado.",
    "Colombres": "Integra el aglomerado Delfín Gallo–Colombres–La Florida.",
    "Ingenio La Florida": "INDEC = 'La Florida' (incluye el Ingenio); integra el mismo aglomerado.",
    "Ranchillos": "Gobierno local 'Ranchillos y San Miguel'.",
    "Lastenia": "Verificar si INDEC la reporta como gobierno local propio o dentro de otro.",
}
nuevas["nota"] = nuevas["localidad"].map(notas).fillna("")

tabla = nuevas[["localidad", "gobierno_local_INDEC", "departamento",
                "primer_registro", "poblacion_2022", "rango_tamano", "nota"]]
tabla = tabla.sort_values(["primer_registro", "localidad"]).reset_index(drop=True)

print(tabla.to_string(index=False))
